This is a simple analysis/comparison of free vs. member functions with Synfig's `Color::clamped()` function.

The test is freeing the function (multiple ways) versus the baseline (where it's a member function).  Note that only GCC on Linux (Ubuntu 24.04 LTS) was tested (clang was failing to compile the Synfig code base).  All work is based off of the `v1.5.3` release of Synfig.

In [1]:
import json
from collections import namedtuple
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import zscore
from ipywidgets import FloatSlider

In [24]:
# Listing the sheets manually here
#   Note: their JSON doesn't contain metadata about the compiler, os, environment, so that's added manually by us (but can
#         be found in the file names)
intel_baseline_file_path        = 'synfig_clamped_linux_measurements/intel_gcc_14.2/results_i7_linux_baseline.json'
intel_freind_function_file_path = 'synfig_clamped_linux_measurements/intel_gcc_14.2/results_i7_linux_friend_function.json'
intel_public_members_files_path = 'synfig_clamped_linux_measurements/intel_gcc_14.2/results_i7_linux_public_members.json'
intel_pass_args_file_path       = 'synfig_clamped_linux_measurements/intel_gcc_14.2/results_i7_linux_pass_arguments.json'
amd_baseline_file_path          = 'synfig_clamped_linux_measurements/amd_gcc_14.2/results_baseline_gcc14.2.json'
amd_freind_function_file_path   = 'synfig_clamped_linux_measurements/amd_gcc_14.2/results_friend_function_gcc14.2.json'
amd_public_members_files_path   = 'synfig_clamped_linux_measurements/amd_gcc_14.2/results_public_members_gcc14.2.json'
amd_pass_args_file_path         = 'synfig_clamped_linux_measurements/amd_gcc_14.2/results_pass_arguments_gcc14.2.json'

In [3]:
# Some contsants
CPU_INTEL = 'Intel i7-10750H'
CPU_AMD = 'AMD Ryzen 9 6900HX'

TT_BASELINE = 'baseline'
SUB_TT_MEMBER_FUNCTION = 'member_function'

TT_FREE = 'free'
SUB_TT_FRIEND_FUNCTION = 'friend_function'
SUB_TT_PUBLIC_MEMBERS = 'public_members'
SUB_TT_PASS_ARGUMENTS = 'pass_arguments'        # The "proper" method of of freeing

# Put the test cases in a logical structure
TestSuite = namedtuple('TestSuite', ['cpu', 'test_type', 'sub_test_type', 'results_json_file_path'])
all_test_suites = [
    TestSuite(CPU_INTEL, TT_BASELINE, SUB_TT_MEMBER_FUNCTION, intel_baseline_file_path),
    TestSuite(CPU_INTEL, TT_FREE,     SUB_TT_FRIEND_FUNCTION, intel_freind_function_file_path),
    TestSuite(CPU_INTEL, TT_FREE,     SUB_TT_PUBLIC_MEMBERS,  intel_public_members_files_path),
    TestSuite(CPU_INTEL, TT_FREE,     SUB_TT_PASS_ARGUMENTS,  intel_pass_args_file_path),
    TestSuite(CPU_AMD,   TT_BASELINE, SUB_TT_MEMBER_FUNCTION, amd_baseline_file_path),
    TestSuite(CPU_AMD,   TT_FREE,     SUB_TT_FRIEND_FUNCTION, amd_freind_function_file_path),
    TestSuite(CPU_AMD,   TT_FREE,     SUB_TT_PUBLIC_MEMBERS,  amd_public_members_files_path),
    TestSuite(CPU_AMD,   TT_FREE,     SUB_TT_PASS_ARGUMENTS,  amd_pass_args_file_path),
]

In [4]:
# A "run set" is a collection of runs that test the same cpu, method type, and sub method type
#   For the data we took, they come in packs of 10.
run_set_id = 1

def test_suite_to_data_frame(test_suite: TestSuite, duration_num_decimal_places:int = 2) -> pd.DataFrame:
    '''Takes in the description of a test sutie and returns a data frame of the results.
    Note that the `duration_ns` will be converted to milliseconds (for human readability).
    '''

    global run_set_id

    suite_results = []
    with open(test_suite.results_json_file_path, 'r') as f:
        data = json.load(f)
        results = data['results']

        for test in results:
            # == Verify we ran okay ==
            # All values in `statuses` should be `Success`
            statuses = test['status']
            for status in statuses:
                assert status == 'Success'

            # All values in `return_codes` should be 0
            return_codes = test['return_code']
            for return_code in return_codes:
                assert return_code == 0

            # == Extract the data we want ==
            id_num = test['id_num']
            sif_filename = Path(test['filepath']).name
            durations_ns = test['duration_nanoseconds']

            # Convert to milliseconds (for human readability)
            durations_ms = [round(d / 1_000_000, duration_num_decimal_places) for d in durations_ns]
            num_runs = len(durations_ms)

            # Place into data frame
            run_set = pd.DataFrame({
                'cpu': [test_suite.cpu] * num_runs,
                'test_type': [test_suite.test_type] * num_runs,
                'sub_test_type': [test_suite.sub_test_type] * num_runs,
                'id_num': [id_num] * num_runs,
                'sif_filename': [sif_filename] * num_runs,
                'run_set_id': [run_set_id] * num_runs,
                'duration_ms': durations_ms,
            })
            suite_results.append(run_set)
            run_set_id += 1

    return pd.concat(suite_results)

#display(test_suite_to_data_frame(all_test_suites[0]))

In [5]:
# Put all of the test suites into a single table
all_test_suites_df = pd.concat([test_suite_to_data_frame(ts) for ts in all_test_suites])
all_test_suites_df.reset_index(inplace=True)
all_test_suites_df.drop(columns=['index'], inplace=True)
display(all_test_suites_df)

,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms
0,Intel i7-10750H,baseline,member_function,1,001-geometry-animation.sif,1,1116.30
1,Intel i7-10750H,baseline,member_function,1,001-geometry-animation.sif,1,1166.56
2,Intel i7-10750H,baseline,member_function,1,001-geometry-animation.sif,1,1116.33
3,Intel i7-10750H,baseline,member_function,1,001-geometry-animation.sif,1,1116.28
4,Intel i7-10750H,baseline,member_function,1,001-geometry-animation.sif,1,1116.38
...,...,...,...,...,...,...,...
54395,AMD Ryzen 9 6900HX,free,pass_arguments,680,warp-noise-gradient-2point-noclip.sif,5440,88582.12
54396,AMD Ryzen 9 6900HX,free,pass_arguments,680,warp-noise-gradient-2point-noclip.sif,5440,88380.32
54397,AMD Ryzen 9 6900HX,free,pass_arguments,680,warp-noise-gradient-2point-noclip.sif,5440,88437.98
54398,AMD Ryzen 9 6900HX,free,pass_arguments,680,warp-noise-gradient-2point-noclip.sif,5440,90477.71


For all eight of the test suite runs, lets see what their cumulative times were.

In [6]:
test_suite_columns = ['cpu', 'test_type', 'sub_test_type']  # columns we need to group for a test sutie

# Convert the `duration_ms` column to data to something human readable
def duration_str(time_ms):
    total_s = time_ms / 1000.0
    hours = int(total_s // 3600)
    minutes = int((total_s % 3600) // 60)
    seconds = int(total_s % 60)
    return f'{hours}h {minutes}m {seconds}s'


def make_test_suite_summary(df: pd.DataFrame) -> pd.DataFrame:
    baseline_time_ms = 0
    suite_durations = []
    for _, group in df.groupby(test_suite_columns):
        # Create the new human readable time entry
        total_time_ms = group['duration_ms'].sum()
        total_time_str = duration_str(total_time_ms)

        # Record the baseline
        is_baseline = group['test_type'].iloc[0] == TT_BASELINE
        if is_baseline:
            baseline_time_ms = total_time_ms

        # Figure out how different it is from the baseline
        from_baseline_delta_ms = total_time_ms - baseline_time_ms
        from_baseline_delta_ratio = from_baseline_delta_ms / baseline_time_ms
        from_baseline_delta_str = f'{round(from_baseline_delta_ratio * 100, 2)}%'

        # Create the new row
        row =  group.iloc[0].copy()
        row['duration_ms'] = total_time_ms
        row['total_duration'] = total_time_str
        row['duration_difference'] = from_baseline_delta_str if not is_baseline else ''
        suite_durations.append(row)

    return pd.DataFrame(suite_durations)

def display_test_suite_summary(df: pd.DataFrame) -> None:
    display(df.groupby(test_suite_columns).agg({'duration_ms': 'sum', 'total_duration': 'first', 'duration_difference': 'first'}))

In [7]:
# == Find the total time taken for each test suite run ==

# Remove the columns `id_num` & `sif_filename` (as we dont need that for this)
all_runs_duration = all_test_suites_df.drop(columns=['sif_filename'])

suite_durations = make_test_suite_summary(all_runs_duration)
display_test_suite_summary(suite_durations)

duration_ms total_duration  \
cpu                test_type sub_test_type                                 
AMD Ryzen 9 6900HX baseline  member_function  32252652.95     8h 57m 32s   
                   free      friend_function  32313431.29     8h 58m 33s   
                             pass_arguments   32305425.02     8h 58m 25s   
                             public_members   32357121.64     8h 59m 17s   
Intel i7-10750H    baseline  member_function  37720425.71    10h 28m 40s   
                   free      friend_function  37758100.02    10h 29m 18s   
                             pass_arguments   37529081.01    10h 25m 29s   
                             public_members   37816829.61    10h 30m 16s   

                                             duration_difference  
cpu                test_type sub_test_type                        
AMD Ryzen 9 6900HX baseline  member_function                      
                   free      friend_function               0.19%  
                             pass_arguments                0.16%  
                             public_members                0.32%  
Intel i7-10750H    baseline  member_function                      
                   free      friend_function                0.1%  
                             pass_arguments               -0.51%  
                             public_members                0.26%

We want to have a lower `duration_ms` and `total_duration`.

A negative percentage (and a greater one) is what we desire to see in the `duration_difference` column.

Looking at the cumulative times, we can see there isn't much of a difference when using a free functions vs the baseline.  Checking the  `duration_difference` column, consider anything that's outside the range of `(-1.0%, 1.0%)` as significant.  These differences are falling between `(-0.5%, 0.3%)`.  This is not a performance hit or gain, **it is simply noise**.

---

What if take the best run of each test (run set) instead?

In [8]:
test_case_columns = ['cpu', 'test_type', 'sub_test_type', 'id_num', 'sif_filename']  # columns we need to group for a test case

def find_best_runs(df: pd.DataFrame) -> pd.DataFrame:
    runs = []
    for _, group in df.groupby(test_case_columns):
        # In this group, get the row that has the lowest `duration_ms`
        best = group.loc[group['duration_ms'].idxmin()].to_frame().T
        runs.append(best)

    runs = pd.concat(runs)
    runs.reset_index(inplace=True)
    runs.drop(columns=['index'], inplace=True)

    return runs

best_runs = find_best_runs(all_test_suites_df)
display(best_runs)

,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms
0,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1066.81
1,AMD Ryzen 9 6900HX,baseline,member_function,2,002-image.sif,2722,465.17
2,AMD Ryzen 9 6900HX,baseline,member_function,3,003-image-group.sif,2723,464.59
3,AMD Ryzen 9 6900HX,baseline,member_function,4,004-image-group-cutout.sif,2724,1366.73
4,AMD Ryzen 9 6900HX,baseline,member_function,5,005-vectors.sif,2725,414.54
...,...,...,...,...,...,...,...
5435,Intel i7-10750H,free,public_members,676,warp-checkerboard-x2-1point-clip.sif,2036,4773.71
5436,Intel i7-10750H,free,public_members,677,warp-noise-gradient-1point-clip.sif,2037,12789.62
5437,Intel i7-10750H,free,public_members,678,warp-noise-gradient-1point-noclip.sif,2038,14292.0
5438,Intel i7-10750H,free,public_members,679,warp-noise-gradient-2point-clip.sif,2039,109904.12


In [9]:
#== Find the duration but only with the best case run times ==#
suite_durations = make_test_suite_summary(best_runs)
display_test_suite_summary(suite_durations)

duration_ms total_duration  \
cpu                test_type sub_test_type                                 
AMD Ryzen 9 6900HX baseline  member_function   3185079.76      0h 53m 5s   
                   free      friend_function   3192055.24     0h 53m 12s   
                             pass_arguments    3191485.83     0h 53m 11s   
                             public_members    3199565.28     0h 53m 19s   
Intel i7-10750H    baseline  member_function   3748494.54      1h 2m 28s   
                   free      friend_function   3752034.88      1h 2m 32s   
                             pass_arguments    3729006.31       1h 2m 9s   
                             public_members    3755963.57      1h 2m 35s   

                                             duration_difference  
cpu                test_type sub_test_type                        
AMD Ryzen 9 6900HX baseline  member_function                      
                   free      friend_function               0.22%  
                             pass_arguments                 0.2%  
                             public_members                0.45%  
Intel i7-10750H    baseline  member_function                      
                   free      friend_function               0.09%  
                             pass_arguments               -0.52%  
                             public_members                 0.2%

Using the best case for each test, we have practically the same results here (minus a few decimal places).  Once again, nothing significant **only noise**.

---

Lastly, let's see if any individual test cases had a significant performance boost difference from being free (or non-free)

In [10]:
# Anything that's about 2% faster is significant
min_percent_faster_threshold = 0.02

def find_significant_runs(df: pd.DataFrame, min_threshold: float=0.02) -> pd.DataFrame:
    runs = []
    for _, group in df.groupby(['cpu', 'id_num']):
        # Get the baseline and the best run of the "frees",
        group.reset_index(inplace=True)
        baseline_run = group.loc[group['test_type'] == TT_BASELINE]
        free_runs = group.loc[group['test_type'] == TT_FREE]
        best_free_run = free_runs.loc[free_runs['duration_ms'].idxmin()]

        # Figure out if there is a significant difference between the two numbers
        baseline_ms = baseline_run['duration_ms'].iloc[0]
        free_ms = best_free_run['duration_ms']
        percent_faster = abs(baseline_ms - free_ms) / baseline_ms

        if percent_faster > min_threshold:
            # Find if the baseline or the free funciton was faster
            performance_difference_ms = 0.0
            best_run = None
            if baseline_ms < free_ms:
                best_run = baseline_run
                performance_difference_ms = free_ms - baseline_ms
            else:
                best_run = best_free_run.to_frame().T
                performance_difference_ms = baseline_ms -free_ms

            best_run_annotated = best_run.copy()
            best_run_annotated['performance_difference_ms'] = performance_difference_ms
            best_run_annotated['percent_faster'] = percent_faster
            best_run_annotated.drop(columns=['index'], inplace=True)

            runs .append(best_run_annotated)

    runs = pd.concat(runs)
    runs.reset_index(inplace=True)
    runs.drop(columns=['index'], inplace=True)

    # Sort by the `percent_faster` column
    runs.sort_values(by=['percent_faster'], inplace=True, ascending=False)

    # Format to look nicer
    runs['percent_faster'] = runs['percent_faster'].apply(lambda x: f'{round(x * 100, 2)}%')

    return runs


significant_runs = find_significant_runs(best_runs, min_percent_faster_threshold)

num_faster = len(significant_runs)
print(f'Found {num_faster} significant runs (above {min_percent_faster_threshold * 100}% faster)')

# Look at some of the most performannt
display_n_top_runs = 10
print(f'Top {display_n_top_runs} most performant runs:')
display(significant_runs.head(display_n_top_runs))

Found 129 significant runs (above 2.0% faster)
Top 10 most performant runs:


,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms,performance_difference_ms,percent_faster
37,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,114.02,50.50,44.29%
32,AMD Ryzen 9 6900HX,free,friend_function,207,layer_gradient_noise_icon.sif,3607,114.08,50.59,30.72%
28,AMD Ryzen 9 6900HX,free,friend_function,180,layer_distortion_curvewarp_icon.sif,3580,114.09,50.42,30.65%
27,AMD Ryzen 9 6900HX,free,friend_function,178,layer_blur_motion_icon.sif,3578,114.17,50.43,30.64%
34,AMD Ryzen 9 6900HX,free,friend_function,241,set_outline_color.sif,3641,114.15,50.36,30.61%
36,AMD Ryzen 9 6900HX,free,friend_function,295,type_canvas_icon.sif,3695,114.23,50.22,30.54%
31,AMD Ryzen 9 6900HX,free,friend_function,201,layer_geometry_region_icon.sif,3601,114.32,50.25,30.53%
33,AMD Ryzen 9 6900HX,free,friend_function,220,layer_other_xorpattern_icon.sif,3620,114.37,50.23,30.52%
30,AMD Ryzen 9 6900HX,free,pass_arguments,199,layer_geometry_polygon_icon.sif,4959,114.43,50.17,30.48%
29,AMD Ryzen 9 6900HX,free,friend_function,190,layer_filter_halftone2_icon.sif,3590,114.28,50.08,30.47%


So this is surprising...  We have some runs that are **significantly** more performant.  I'm seeing 30%+ improvements?  With one being 44% faster (and as a member function)?!  Something seems a little odd.  Let's look at the raw test data for the best performing one.

In [11]:
# Get top run's entire "run set" it's from
biggest_difference_run = significant_runs.iloc[0]
biggest_difference_run_set = all_test_suites_df.loc[all_test_suites_df['run_set_id'] == biggest_difference_run['run_set_id']]

# Show all of the runs so we can inspect the data
original_max_rows = pd.get_option('display.max_rows')
pd.set_option('display.max_rows', None)
display(biggest_difference_run_set)
pd.set_option('display.max_rows', original_max_rows)

,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms
30150,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.59
30151,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.89
30152,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.54
30153,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.51
30154,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.61
30155,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.48
30156,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.61
30157,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.50
30158,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,114.02
30159,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,164.28


Most of the values are hoving around `164 ms`, with one reporting a runtime of `114 ms`.  **This is a satiscial outlier and throwing off our analysis.**  Using something called "Z-Scores" can help us identify the rows that are statistical outliers, and then we can throw them out.

In [12]:
# Function to highlight rows with z-scores above the threshold
def highlight_outliers(row):
    z_score_threshold = 1.0 # Use 1.0 as the threshold for coloring as requested
    if abs(row['zscore']) > z_score_threshold:
        return ['color: red'] * len(row)
    return [''] * len(row)

# Identify outliers based on the z-score threshold
z_scores = zscore(biggest_difference_run_set['duration_ms'])

biggest_difference_run_set_with_zscores = biggest_difference_run_set.copy()
biggest_difference_run_set_with_zscores['zscore'] = z_scores

display(biggest_difference_run_set_with_zscores[['duration_ms', 'zscore']].style.apply(highlight_outliers, axis=1))
print(f'\n Variance: {np.var(biggest_difference_run_set_with_zscores["duration_ms"]):.2f}')

,duration_ms,zscore
30150,164.590000,0.335517
30151,164.890000,0.355304
30152,164.540000,0.332219
30153,164.510000,0.330240
30154,164.610000,0.336836
30155,164.480000,0.328262
30156,164.610000,0.336836
30157,164.500000,0.329581
30158,114.020000,-2.999866
30159,164.280000,0.315071



 Variance: 229.88


In [13]:
# Adjustable z-score threshold slider
z_score_threshold_widget = FloatSlider(min=1.0, max=3.0, step=0.1, value=2.0, description='Z-score Threshold:')
display(z_score_threshold_widget)

FloatSlider(value=2.0, description='Z-score Threshold:', max=3.0, min=1.0)

In [14]:
z_score_threshold = z_score_threshold_widget.value
print(f'Z-score threshold: {z_score_threshold}')

Z-score threshold: 2.0


In [15]:
# Identify outliers based on the z-score threshold
z_scores = zscore(biggest_difference_run_set['duration_ms'])
outliers_zscore = biggest_difference_run_set[(z_scores > z_score_threshold) | (z_scores < -z_score_threshold)]

print(f'Outliers based on Z-score threshold of {z_score_threshold}:')
display(outliers_zscore)

Outliers based on Z-score threshold of 2.0:


,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms
30158,AMD Ryzen 9 6900HX,baseline,member_function,296,type_color_icon.sif,3016,114.02


In [16]:
# For some extra validation, we can look at the variance for the `duration_ms` column of the inliers
inliers_zscore = biggest_difference_run_set[(z_scores <= z_score_threshold) & (z_scores >= -z_score_threshold)]
variance_inliers = np.var(inliers_zscore['duration_ms'])
print(f'Variance of inliers: {variance_inliers:.2f}')

Variance of inliers: 0.02


In [17]:
# Remove the identified outliers from the main DataFrame
# We will use the z-score method for outlier detection as requested by the user.
def remove_outliers_zscore(df: pd.DataFrame, group_cols: list, threshold: float = 2.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''
    Removes outliers from a DataFrame based on the z-score of a specified column within groups.

    Args:
        df: The input DataFrame.
        group_cols: A list of column names to group by before identifying outliers.
        threshold: The z-score threshold for identifying outliers.

    Returns:
        A tuple containing two DataFrames:
        - The DataFrame with outliers removed.
        - The DataFrame containing only the identified outliers.
    '''
    df_cleaned = pd.DataFrame()
    df_outliers = pd.DataFrame()

    for _, group in df.groupby(group_cols):
        z_scores = zscore(group['duration_ms'])
        outlier_mask = (z_scores > threshold) | (z_scores < -threshold)

        df_cleaned = pd.concat([df_cleaned, group[~outlier_mask]])
        df_outliers = pd.concat([df_outliers, group[outlier_mask]])

    return df_cleaned, df_outliers


# Remove outliers using the z-score method
all_test_suites_df_without_outliers, identified_outliers = remove_outliers_zscore(
    all_test_suites_df,
    ['cpu', 'test_type', 'sub_test_type', 'id_num'],
    z_score_threshold # Use the threshold defined in the previous cell
)

In [18]:
print('DataFrame with outliers removed:')
display(all_test_suites_df_without_outliers)

print('Identified outliers:')
display(identified_outliers)

DataFrame with outliers removed:


,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms
27201,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1067.36
27202,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1066.81
27203,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1167.57
27204,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1117.75
27205,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1167.18
...,...,...,...,...,...,...,...
20395,Intel i7-10750H,free,public_members,680,warp-noise-gradient-2point-noclip.sif,2040,109611.57
20396,Intel i7-10750H,free,public_members,680,warp-noise-gradient-2point-noclip.sif,2040,110619.44
20397,Intel i7-10750H,free,public_members,680,warp-noise-gradient-2point-noclip.sif,2040,109463.32
20398,Intel i7-10750H,free,public_members,680,warp-noise-gradient-2point-noclip.sif,2040,111022.53


Identified outliers:


,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms
27200,AMD Ryzen 9 6900HX,baseline,member_function,1,001-geometry-animation.sif,2721,1623.91
27229,AMD Ryzen 9 6900HX,baseline,member_function,3,003-image-group.sif,2723,464.59
27242,AMD Ryzen 9 6900HX,baseline,member_function,5,005-vectors.sif,2725,465.78
27270,AMD Ryzen 9 6900HX,baseline,member_function,8,008-skeleton.sif,2728,715.83
27281,AMD Ryzen 9 6900HX,baseline,member_function,9,009-skeleton.sif,2729,665.37
...,...,...,...,...,...,...,...
20310,Intel i7-10750H,free,public_members,672,warp-checkerboard-1point-clip.sif,2032,3722.32
20336,Intel i7-10750H,free,public_members,674,warp-checkerboard-2point-clip.sif,2034,16746.07
20340,Intel i7-10750H,free,public_members,675,warp-checkerboard-2point-noclip.sif,2035,21457.79
20374,Intel i7-10750H,free,public_members,678,warp-noise-gradient-1point-noclip.sif,2038,14292.00


In [19]:
# Calculuate what % of data we have thrown out
num_total = len(all_test_suites_df)
num_okay = len(all_test_suites_df_without_outliers)
num_outliers = len(identified_outliers)

percent_thrown_out = num_outliers / num_total
print(f'Percent of data thrown out: {round(percent_thrown_out * 100, 2)}%')

Percent of data thrown out: 5.26%


While this isn't an ideal amount of data we've had to toss , let's re-run the analysis with newer cleaned data.

In [20]:
cleand_suite_durations = make_test_suite_summary(all_test_suites_df_without_outliers)
display_test_suite_summary(cleand_suite_durations)

duration_ms total_duration  \
cpu                test_type sub_test_type                                 
AMD Ryzen 9 6900HX baseline  member_function  31048185.86     8h 37m 28s   
                   free      friend_function  30180460.30      8h 23m 0s   
                             pass_arguments   30874040.25     8h 34m 34s   
                             public_members   30412491.95     8h 26m 52s   
Intel i7-10750H    baseline  member_function  35923254.31     9h 58m 43s   
                   free      friend_function  35840601.86     9h 57m 20s   
                             pass_arguments   35577778.80     9h 52m 57s   
                             public_members   35362490.47     9h 49m 22s   

                                             duration_difference  
cpu                test_type sub_test_type                        
AMD Ryzen 9 6900HX baseline  member_function                      
                   free      friend_function              -2.79%  
                             pass_arguments               -0.56%  
                             public_members               -2.05%  
Intel i7-10750H    baseline  member_function                      
                   free      friend_function              -0.23%  
                             pass_arguments               -0.96%  
                             public_members               -1.56%

This is now a little surprising.  When we threw out some of the statistical outliers, we found that running the free version(s) of the `Color::clamped()` actually was a small bit faster, but with a significant margin!  But here are some observations:

- For AMD, doing `pass_arguments` (the "proper" method) of freeing wasn't that much faster and still in the "noise" category.  But using a friend function or having the data as a public member had a 2% performance boost!
- For Intel, friend functions are in the realm of noise, `pass_arguments` is borderline, and public data members are a bit better.


Let's look at the best base run per test.

In [21]:
cleaned_best_runs = find_best_runs(all_test_suites_df_without_outliers)
cleaned_best_runs_summary = make_test_suite_summary(cleaned_best_runs)
display_test_suite_summary(cleaned_best_runs_summary)

duration_ms total_duration  \
cpu                test_type sub_test_type                                 
AMD Ryzen 9 6900HX baseline  member_function   3193037.24     0h 53m 13s   
                   free      friend_function   3201728.96     0h 53m 21s   
                             pass_arguments    3199456.47     0h 53m 19s   
                             public_members    3207045.12     0h 53m 27s   
Intel i7-10750H    baseline  member_function   3755393.25      1h 2m 35s   
                   free      friend_function   3757140.80      1h 2m 37s   
                             pass_arguments    3737260.68      1h 2m 17s   
                             public_members    3763326.10      1h 2m 43s   

                                             duration_difference  
cpu                test_type sub_test_type                        
AMD Ryzen 9 6900HX baseline  member_function                      
                   free      friend_function               0.27%  
                             pass_arguments                 0.2%  
                             public_members                0.44%  
Intel i7-10750H    baseline  member_function                      
                   free      friend_function               0.05%  
                             pass_arguments               -0.48%  
                             public_members                0.21%

When we look at the best case though for each test, the performance change is now back to the range of being "just noise".  Let's inspect if there are cases where one method is at least 2% faster than the others (using the "cleaned best case" runs).

In [22]:
 # Find if there are any test cases that are signficantly more performant (see above)
cleaned_significant_runs = find_significant_runs(cleaned_best_runs, min_percent_faster_threshold)

num_faster = len(cleaned_significant_runs)
print(f'Found {num_faster} significant runs (above {min_percent_faster_threshold * 100}% faster)')

pd.set_option('display.max_rows', None)
display(cleaned_significant_runs)
pd.set_option('display.max_rows', original_max_rows)

Found 164 significant runs (above 2.0% faster)


,cpu,test_type,sub_test_type,id_num,sif_filename,run_set_id,duration_ms,performance_difference_ms,percent_faster
50,AMD Ryzen 9 6900HX,baseline,member_function,300,type_real_icon.sif,3020,114.03,50.28,44.09%
46,AMD Ryzen 9 6900HX,free,friend_function,208,layer_gradient_radial_icon.sif,3608,113.94,50.53,30.72%
43,AMD Ryzen 9 6900HX,free,pass_arguments,204,layer_gradient_conical_icon.sif,4964,114.01,50.51,30.7%
45,AMD Ryzen 9 6900HX,free,friend_function,207,layer_gradient_noise_icon.sif,3607,114.17,50.50,30.67%
41,AMD Ryzen 9 6900HX,free,friend_function,180,layer_distortion_curvewarp_icon.sif,3580,114.11,50.40,30.64%
40,AMD Ryzen 9 6900HX,free,friend_function,178,layer_blur_motion_icon.sif,3578,114.2,50.40,30.62%
42,AMD Ryzen 9 6900HX,free,friend_function,200,layer_geometry_rectangle_icon.sif,3600,114.16,50.31,30.59%
39,AMD Ryzen 9 6900HX,free,pass_arguments,147,action_remove_from_set_icon.sif,4907,114.17,50.28,30.57%
49,AMD Ryzen 9 6900HX,free,friend_function,295,type_canvas_icon.sif,3695,114.23,50.22,30.54%
47,AMD Ryzen 9 6900HX,free,friend_function,241,set_outline_color.sif,3641,114.3,50.21,30.52%


The test suites look to have precise (good) data, but also some with a lot of statistical outliers (so more variation in the data).  Let's isolate one of the "run sets" to inspect the data a little more.

In [23]:
# Isolate run set 754; just print the durations:
runs_754 = all_test_suites_df.loc[all_test_suites_df['run_set_id'] == 754]
durations_754 = runs_754['duration_ms']
zscores_754 = zscore(durations_754)
variance_754 = np.var(durations_754)

# Attach the zcores as a new column
run_set_754_with_zscores = runs_754.copy()
run_set_754_with_zscores['zscore'] = zscores_754

# Display the duration, along with the zscore for each run set
display(run_set_754_with_zscores[['duration_ms', 'zscore']])
print(f'\nVariance: {variance_754:.2f}')

,duration_ms,zscore
7530,515.23,1.001346
7531,515.12,0.996985
7532,464.71,-1.001425
7533,515.18,0.999363
7534,464.75,-0.999839
7535,515.22,1.000949
7536,464.71,-1.001425
7537,515.23,1.001346
7538,464.69,-1.002218
7539,464.87,-0.995082



Variance: 636.30


Taking a look at this, the variance is quite high, and the zscore is flip flopping between (around) `+1` and `-1`.  With about 50% on each side of the zscore.  **This unfortunately means the "run set" is not a good test**.

---

To make comparisons across CPU/OS/Compiler, I think it's acceptable to throw one data point (maybe two) from a run set.  But if something like this run set (no. 754) has poor data, I'd feel obligated to throw the run sets that use the same .sif file (`075-ATF-skeleton-group.sif`), for the other test cases.  E.g. if "member funciton's" is bad, then we can't compare it to any of the free funcitons".  Same if any of the freeing methods had bad data.  By doing this, it could lead to a lot of data being thrown out.

Above we used a Z-Score threshold of `2.0`, and we had to throw out about 5% of our data.  I took at look at some of the other runs sets.  Some had low variance (with convergent z-scores for all runs), and others were quite high (and divergent z-scores).  Adjusting this threshold can lead to even more data being thrown out.  When I tried setting it to `1.0` (very sensitive), 30% of the data was then thrown out!!

When I tinkered with the z-score more and more, I saw there were best case scenarios where free functions were in the lead, and others where member functions were doing better.

An attempt was also made using other methods such as IQR to identify bad data, but that also threw out too much.

With doing about ten runs per test case, but seeing some have some very imprecise data, I don't even think that performing more runs per test case would even be beneficial.

---

**My conclusion from this is that the performance of a free function is indistinguishable from that of a member function (in a larger application).**  I'm hesitant to say one is better over the other, in terms of performance.